# Group Work 3：線形代数による乳がん分類

## 目的

30次元の乳がんデータに対し，線形最小二乗分類器を実装する．勾配降下法とQR分解を用いて仮説ベクトルを求め，未知データで評価する．

## 提出物とルール

- このノートブックの`Your code goes here.`と考察欄を埋め，**上から順に全セルを実行した状態**で提出する．
- 数値だけでなく，shape，損失，混同行列，文章による説明も評価対象とする．
- 参考資料は`gw3_breast_cancer.ipynb`，線形代数の復習は`linear_algebra_scipy.ipynb`を用いる．
- 生成AIや他者のコードを利用した場合は，末尾に利用箇所と検証方法を記す．

データは授業用に分割済みであり，悪性を$+1$，良性を$-1$とする．この課題のモデルは学習用であり，医療診断には使用できない．

## 0．準備

次のセルは変更しない．`wdbc_data`ディレクトリから，訓練データ300件と検証データ269件を読み込む．

In [ ]:
from pathlib import Path
import numpy as np

np.set_printoptions(precision=5, suppress=True)
DATA_DIR = Path("wdbc_data")

def load_split(name, n_rows):
    x = np.loadtxt(DATA_DIR / f"{name}_A.data", dtype=float).reshape(n_rows, 30)
    y = np.loadtxt(DATA_DIR / f"{name}_b.data", dtype=float).reshape(n_rows, 1)
    return x, y

x_train, y_train = load_split("train", 300)
x_valid, y_valid = load_split("validate", 269)
print("train:", x_train.shape, y_train.shape)
print("valid:", x_valid.shape, y_valid.shape)

FEATURE_NAMES = [
    "radius", "texture", "perimeter", "area", "smoothness",
    "compactness", "concavity", "concave_pts", "symmetry", "fractal_dim",
]


def print_w(w):
    """30次元の係数を，特徴量ごとにmean，StdErr，worstの順で表示する．"""
    values = np.asarray(w, dtype=float).reshape(-1)
    if values.size != 30:
        raise ValueError("w must contain 30 elements")

    print(f'{"feature":<14}{"mean":>13}{"StdErr":>13}{"worst":>13}')
    print("-" * 53)
    for i, name in enumerate(FEATURE_NAMES):
        print(f"{name:<14}{values[i]:13.6f}{values[i + 10]:13.6f}{values[i + 20]:13.6f}")


def _classification_counts(mA, vb, vw):
    actual = np.asarray(vb).reshape(-1)
    scores = (np.asarray(mA) @ np.asarray(vw)).reshape(-1)
    predicted = np.where(scores >= 0, 1, -1)

    tp = int(np.sum((actual == 1) & (predicted == 1)))
    tn = int(np.sum((actual == -1) & (predicted == -1)))
    fp = int(np.sum((actual == -1) & (predicted == 1)))
    fn = int(np.sum((actual == 1) & (predicted == -1)))
    total = actual.size
    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "correct": tp + tn,
        "total": total,
        "accuracy": (tp + tn) / total,
    }


def show_accuracy(mA, vb, vw):
    """正解数，accuracy，偽陽性，偽陰性を簡潔に表示する．"""
    result = _classification_counts(mA, vb, vw)
    print(f"correct       : {result['correct']:4d} / {result['total']:4d}")
    print(f"accuracy      : {result['accuracy']:8.3%}")
    print(f"false positive: {result['fp']:4d}  (良性を悪性と予測)")
    print(f"false negative: {result['fn']:4d}  (悪性を良性と予測)")
    return result


def show_accuracy2(mA, vb, vw):
    """混同行列を表示する．positiveは悪性（+1）である．"""
    result = _classification_counts(mA, vb, vw)
    print(" " * 22 + "predicted")
    print(f'{"":22}{"malignant(+1)":>15}{"benign(-1)":>13}')
    print(f'{"actual malignant(+1)":22}{result["tp"]:15d}{result["fn"]:13d}')
    print(f'{"actual benign(-1)":22}{result["fp"]:15d}{result["tn"]:13d}')
    print(f"accuracy: {result['accuracy']:.3%}")
    return result

## 1．データとshape

1．`x_train`，`y_train`のshapeを記せ．各軸が何を表すか説明せよ．  
2．ラベルに含まれる値と，それぞれの件数を求めよ．  
3．30要素すべてが`0.0001`である列ベクトル`w0`を作り，`x_train @ w0`のshapeを確かめよ．

In [ ]:
# Your code goes here.

**説明：**  
（ここに各shapeの意味と，ラベルの集計結果を書く．）

## 2．スコアと損失

損失を

$$
J(w)=\frac{1}{2}\lVert Aw-b\rVert_2^2
$$

とする．次を行え．

1．任意の`w`に対して損失を返す関数`squared_loss(A, b, w)`を実装せよ．  
2．`w0`に対する訓練データの損失を求めよ．  
3．最初の10件について，スコア，予測ラベル，正解ラベルを表示せよ．スコアが0以上なら$+1$，0未満なら$-1$とする．

In [ ]:
# Your code goes here.

**考察：**  
（損失と分類の正解数が同じ意味でない理由を，スコアの値に触れて説明する．）

## 3．勾配と1回の更新

$$
\nabla J(w)=A^\mathsf{T}(Aw-b)
$$

を用いる．

1．任意の`w`に対して勾配を返す関数`loss_gradient(A, b, w)`を実装せよ．  
2．`w0`で勾配を求め，そのshapeが`w0`と等しいことを確認せよ．  
3．学習率`eta = 3.0e-9`で1回更新した`w1`を求め，更新前後の損失を比較せよ．

In [ ]:
# Your code goes here.

**説明：**  
（勾配のshapeが`w`と同じになる理由を，行列積のshapeを示して説明する．）

## 4．勾配降下法

`w0`から出発し，300回更新したベクトル`w_gd`を求めよ．各反復の更新後の損失をリスト`loss_history`に保存すること．

さらに，次を報告せよ．

- 初期損失と最終損失
- 1，10，50，100，300回目の損失
- `print_w(w_gd)`による係数一覧
- `show_accuracy(x_train, y_train, w_gd)`による訓練結果

`loss_history`が300要素であること，すべて有限値であることも確認せよ．

In [ ]:
# Your code goes here.

**考察：**  
（損失の推移から収束の様子を説明する．学習率を10倍にした場合に何が起きると予想するかも書く．）

## 5．QR分解による最小二乗解

`np.linalg.qr`のreduced QR分解を用い，`x_train = Q @ R`とせよ．

1．`Q`と`R`のshapeを確認せよ．  
2．`Q.T @ y_train`を計算し，そのshapeと最初の5要素を表示せよ．  
3．`w_qr = np.linalg.solve(R, Q.T @ y_train)`として$Rw=Q^\mathsf{T}b$を解け．逆行列を明示的に作らないこと．  
4．`predicted_b = x_train @ w_qr`を求め，正解`y_train`と予測値`predicted_b`を最初の10件について横に並べて表示せよ．  
5．`w_qr`の損失を求め，`print_w`と`show_accuracy`で結果を表示し，勾配降下法と比較せよ．

In [ ]:
# Your code goes here.

**考察：**  
（300回の勾配降下法とQR分解で，損失や計算方法が異なる理由を書く．）

## 6．未知データでの評価

準備セルで定義済みの`show_accuracy`と`show_accuracy2`を用いる．悪性（$+1$）をpositiveとする．

1．4項目の合計が標本数になることを確認せよ．  
2．`w_gd`と`w_qr`を，訓練データと検証データの両方で評価せよ．要約には`show_accuracy`，混同行列には`show_accuracy2`を使うこと．  
3．どちらのモデルを採用するか，損失，accuracy，FN，FPを根拠に述べよ．

In [ ]:
# Your code goes here.

**最終考察：**  
（訓練結果と検証結果の差，FNとFPの意味，採用するモデルを150〜250字で論じる．また，生の係数の絶対値だけで「重要な特徴」を判断しにくい理由も述べる．）

**外部ツール等の利用申告：**  
（利用していない場合は「なし」と書く．利用した場合は，利用箇所，得た支援，自分で行った検証を書く．）